# Part 1: Foundation Model Exploration

This notebook compares a zero-training Hugging Face sentiment model with the HW2 proactive customer satisfaction model.

Important context from the assignment: the HW2 model predicts satisfaction before the review is written using order and delivery features, so it is proactive. The foundation model reads the review text after the customer writes it, so it is reactive.

In [3]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from common import (
    FEATURES, TARGET, build_order_level_dataset, find_data_path, load_raw_data,
)

pd.set_option('display.max_columns', 100)
DATA_PATH = find_data_path()
print(f'Detected dataset: {DATA_PATH.name}')

Detected dataset: olist_cleaned_integrated_dataset (3).csv


## Load Olist Data and Check Review Text Availability

The assignment requires `review_comment_message`. The cleaned HW2/HW3 modeling dataset in this folder contains `review_score_mean` and review dates, but it does not contain the text field. The code below supports either case: if a review-text file is supplied through `REVIEW_TEXT_PATH`, it merges that file by `order_id`; otherwise it reports the limitation clearly instead of inventing review text.

In [4]:
raw = load_raw_data(DATA_PATH)
print(raw.shape)
print([c for c in raw.columns if 'review' in c.lower() or 'comment' in c.lower() or 'message' in c.lower()])

review_text_path = "olist_order_reviews_dataset.csv"
if 'review_comment_message' not in raw.columns and review_text_path:
    review_text = pd.read_csv(review_text_path)
    keep_cols = [c for c in ['order_id', 'review_score', 'review_comment_message'] if c in review_text.columns]
    raw = raw.merge(review_text[keep_cols], on='order_id', how='left')

text_col = 'review_comment_message' if 'review_comment_message' in raw.columns else None
if text_col is None:
    print('No review_comment_message column is available in the detected cleaned modeling dataset.')
else:
    print(raw[text_col].notna().sum(), 'records have non-null review text')

(113425, 41)
['review_score_mean', 'review_creation_date']
48166 records have non-null review text


## Build the Same HW2 Order-Level Features

This reuses the HW2 aggregation, target, feature engineering, and serialized Random Forest pipeline.

In [5]:
agg = build_order_level_dataset(raw)
print(agg.shape)
print('Positive rate:', agg[TARGET].mean())
agg[FEATURES + [TARGET, 'review_score_mean']].head()

(99441, 26)
Positive rate: 0.7613258112850836


,delivery_days,delivery_vs_estimated,total_price,total_freight,total_order_value,log_total_order_value,n_items,avg_item_price,avg_weight_g,avg_volume_cm3,order_hour,order_dayofweek,approval_hours,freight_share,seller_state_match,is_repeat_customer,is_weekend_purchase,payment_value_total,product_category,seller_state,payment_type,is_positive_review,review_score_mean
0,7.0,-9.0,58.90,13.29,72.19,4.293059,1.0,58.90,650.0,3528.0,8,2,0.775833,0.184098,0,0,0,72.19,cool_stuff,SP,credit_card,1,5.0
1,16.0,-3.0,239.90,19.93,259.83,5.563869,1.0,239.90,30000.0,60000.0,10,2,0.201944,0.076704,1,1,0,259.83,pet_shop,SP,credit_card,1,4.0
2,7.0,-14.0,199.00,17.87,216.87,5.383899,1.0,199.00,3050.0,14157.0,14,6,0.249722,0.082400,1,0,1,216.87,furniture_decor,MG,credit_card,1,5.0
3,6.0,-6.0,12.99,12.79,25.78,3.287655,1.0,12.99,200.0,2400.0,10,2,0.161944,0.496121,1,0,0,25.78,perfumery,SP,credit_card,1,4.0
4,25.0,-16.0,199.90,18.14,218.04,5.389254,1.0,199.90,3750.0,42000.0,13,5,0.206111,0.083196,0,0,1,218.04,garden_tools,PR,credit_card,1,5.0


## Foundation Model Inference on 500 Non-Empty Reviews

Model required by the assignment: `nlptown/bert-base-multilingual-uncased-sentiment`. It predicts 1-5 star labels from multilingual text. Predicted 4-5 stars are mapped to positive; 1-3 stars are mapped to negative.

In [7]:
if text_col is None:
    foundation_sample = pd.DataFrame()
    print('SKIPPED: The local cleaned dataset has no review text. Supply REVIEW_TEXT_PATH with order_id and review_comment_message to run this cell.')
else:
    from transformers import pipeline

    text_rows = raw[raw[text_col].fillna('').astype(str).str.strip().ne('')].copy()
    text_rows = text_rows.drop_duplicates('order_id')
    foundation_sample = text_rows.sample(n=min(500, len(text_rows)), random_state=42).copy()

    sentiment = pipeline(
        'sentiment-analysis',
        model='nlptown/bert-base-multilingual-uncased-sentiment',
    )

    outputs = sentiment(foundation_sample[text_col].astype(str).tolist(), truncation=True)
    foundation_sample['foundation_label'] = [o['label'] for o in outputs]
    foundation_sample['foundation_confidence'] = [o['score'] for o in outputs]
    foundation_sample['foundation_stars'] = foundation_sample['foundation_label'].str.extract(r'(\d+)').astype(int)
    foundation_sample['foundation_pred'] = (foundation_sample['foundation_stars'] >= 4).astype(int)
    foundation_sample['actual'] = (foundation_sample['review_score_mean'] >= 4).astype(int)

    display(foundation_sample[['order_id', text_col, 'review_score_mean', 'actual', 'foundation_label', 'foundation_confidence', 'foundation_pred']].head())

c:\Users\jfent\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jfent\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jfent\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mod

,order_id,review_comment_message,review_score_mean,actual,foundation_label,foundation_confidence,foundation_pred
44640,ee8e24472be0999597648d72577ae895,Não nada a declarar \r\nNão cumpre o que promete,1.0,0,1 star,0.625350,0
1257,b9b690e003beef9acdde2502dfc565e2,"Linda a boneca, carinha querida,amei",5.0,1,5 stars,0.518247,1
651,4c460fdf49b517270b6258ba2d3c1ef9,"Produto de ótima qualidade, fiquei super feliz...",5.0,1,5 stars,0.758726,1
63152,53ba9277abd171947b33bd9073297827,mt bom o produto,5.0,1,4 stars,0.403597,1
18178,116916cf522fb4386293fb7cfa886e7e,ja comprei algum tempo atraz na shptime e comp...,5.0,1,1 star,0.619527,0


## Class Distribution

In [8]:
if foundation_sample.empty:
    print('Class distribution cannot be calculated without non-empty review text records.')
else:
    class_distribution = pd.DataFrame({
        'actual_count': foundation_sample['actual'].value_counts().sort_index(),
        'foundation_pred_count': foundation_sample['foundation_pred'].value_counts().sort_index(),
    }).rename(index={0: 'negative', 1: 'positive'})
    display(class_distribution)

,actual_count,foundation_pred_count
negative,188,253
positive,312,247


## HW2 Model Predictions on the Same 500 Records

The HW2 model is evaluated on the exact same sampled orders. This is still not an identical business use case: the HW2 model is proactive and uses order features available before a review is written, while the foundation model is reactive and uses the final review text.

In [10]:
model_path = Path('model/model.pkl')
if not model_path.exists():
    print('model/model.pkl not found. Run: python train_and_serialize.py')
elif foundation_sample.empty:
    print('HW2 comparison on the same 500 records requires the review-text sample above.')
else:
    hw2_model = joblib.load(model_path)
    sample_orders = foundation_sample[['order_id', 'actual']].merge(agg, on='order_id', how='left')
    sample_orders['hw2_proba'] = hw2_model.predict_proba(sample_orders[FEATURES])[:, 1]
    sample_orders['hw2_pred'] = (sample_orders['hw2_proba'] >= 0.5).astype(int)
    display(sample_orders[['order_id', 'actual', 'hw2_pred', 'hw2_proba']].head())

,order_id,actual,hw2_pred,hw2_proba
0,ee8e24472be0999597648d72577ae895,0,0,0.039053
1,b9b690e003beef9acdde2502dfc565e2,1,1,0.907986
2,4c460fdf49b517270b6258ba2d3c1ef9,1,1,0.866784
3,53ba9277abd171947b33bd9073297827,1,1,0.813812
4,116916cf522fb4386293fb7cfa886e7e,1,1,0.847240


## Required Comparison Table

In [11]:
def binary_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
    }

if foundation_sample.empty or not Path('model/model.pkl').exists():
    comparison = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        'Foundation Model (review text)': ['Requires review_comment_message'] * 4,
        'Your HW2 Model (order features)': ['Requires same 500 review-text records'] * 4,
    })
else:
    fm = binary_metrics(foundation_sample['actual'], foundation_sample['foundation_pred'])
    hw2 = binary_metrics(sample_orders['actual'], sample_orders['hw2_pred'])
    comparison = pd.DataFrame({
        'Metric': list(fm.keys()),
        'Foundation Model (review text)': list(fm.values()),
        'Your HW2 Model (order features)': [hw2[k] for k in fm.keys()],
    })

display(comparison)

,Metric,Foundation Model (review text),Your HW2 Model (order features)
0,Accuracy,0.802000,0.724000
1,Precision,0.931174,0.696833
2,Recall,0.737179,0.987179
3,F1 Score,0.822898,0.816976


## Reflection

The HW2 model and the foundation model solve related but different operational problems. The HW2 Random Forest is a proactive early-warning model: it uses order, payment, product, and delivery features to estimate whether a customer is likely to leave a positive review before the review text exists. That makes it useful for intervention, routing, service recovery, and monitoring fulfillment quality. The foundation model is reactive: it reads the written review text after the customer has already expressed an opinion, so it is better suited for labeling, triage, summarization, and post-hoc voice-of-customer analysis.

If review text is available, the foundation model may perform strongly because its input directly contains customer sentiment. Its advantage is speed to deployment: it requires no Olist-specific training data and can handle Portuguese text out of the box. The tradeoff is computational cost, dependency on a large transformer download, and weaker control over domain-specific errors. The custom HW2 model requires feature engineering and training, but it is cheaper to serve, aligned to the exact business definition, and available before the review arrives.

In production, I would use both. The HW2 model would flag risky orders proactively, while the foundation model would classify review text after delivery to enrich monitoring labels, audit false positives and false negatives, and create new features for future retraining.